# UC Question Sprint — Verification Notebook
Runs every answer through MANY independent methods and 100 trials.
Only reports answers that are 100% consistent across all methods.

In [ ]:
import pandas as pd, numpy as np, sqlite3, os
BASE = r'C:/Users/Dhiaan Dave/Downloads/UC Admissions Data Challenge-20260830T004919Z-1-001/UC Admissions Data Challenge/Data'
raw = pd.read_csv(os.path.join(BASE,'bay_area_modeling_table.csv'), low_memory=False)
dash = pd.read_csv(os.path.join(BASE,'dashboard_data.csv'), low_memory=False)
eth = pd.read_csv(os.path.join(BASE,'uc_admissions_summary_by_ethnicity.csv'), low_memory=False)
trmaj = pd.read_csv(os.path.join(BASE,'uc_transfer_admission_by_major.csv'), low_memory=False)
disc = pd.read_csv(os.path.join(BASE,'uc_freshman_admission_by_discipline.csv'), low_memory=False)
# force numeric
for df in (raw,dash):
    for c in ['applicants','admits','enrollees','enrolled_ccc','graduates','ag_completers','admit_rate','admit_rate_residual']:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors='coerce')
print('loaded', raw.shape, dash.shape, eth.shape)

In [ ]:
def method_pandas(raw,dash,eth,trmaj,disc):
    a={}
    # Q1
    y=raw[raw.fall_term==2025]
    a['Q1']=round(y[y.campus!='Universitywide'].applicants.sum()/y[y.campus=='Universitywide'].applicants.sum(),2)
    # Q2
    s=dash[(dash.fall_term==2025)&(dash.campus=='Los Angeles')&(dash.school_type=='High Schools (Public)')]
    a['Q2']=round(s.admits.sum()/s.applicants.sum()*100,2)
    # Q3
    d25=disc[disc.fall_term==2025]
    ov=d25[d25.broad_discipline=='All disciplines'].set_index('campus')['admit_rate']
    cs=d25[d25.broad_discipline=='Computer Science'].set_index('campus')['admit_rate']
    pen=(cs-ov).dropna()*100; a['Q3']=pen.idxmin()
    # Q4
    b=trmaj[(trmaj.campus=='Berkeley')&(trmaj.broad_discipline=='Computer Science')&(trmaj.major=='ComputerScience')]
    a['Q4']=round(b.admit_gpa_p75.iloc[0]-b.admit_gpa_p25.iloc[0],2)
    # Q5
    e25=eth[(eth.fall_term==2025)&(eth.campus!='Systemwide')]
    ap=e25[e25.count_type=='App'].pivot_table(index='campus',columns='ethnicity',values='n',aggfunc='sum')
    ad=e25[e25.count_type=='Adm'].pivot_table(index='campus',columns='ethnicity',values='n',aggfunc='sum')
    a['Q5']=int(((ad['White']/ap['White'])>(ad['Hispanic/Latino(a)']/ap['Hispanic/Latino(a)'])).sum())
    # Q6
    sysw=eth[(eth.fall_term==2025)&(eth.campus=='Systemwide')]
    wr=sysw[(sysw.ethnicity=='White')&(sysw.count_type=='Adm')]['n'].iloc[0]/sysw[(sysw.ethnicity=='White')&(sysw.count_type=='App')]['n'].iloc[0]
    hr=sysw[(sysw.ethnicity=='Hispanic/Latino(a)')&(sysw.count_type=='Adm')]['n'].iloc[0]/sysw[(sysw.ethnicity=='Hispanic/Latino(a)')&(sysw.count_type=='App')]['n'].iloc[0]
    a['Q6']='Hispanic/Latino(a)' if hr>wr else 'White'
    # Q7
    bay=['Alameda','Contra Costa','Marin','Napa','San Francisco','San Mateo','Santa Clara','Solano','Sonoma']
    sub=raw[(raw.fall_term==2023)&(raw.campus=='Universitywide')&(raw.county.isin(bay))]
    a['Q7']=round(sub.enrolled_ccc.sum()/sub.graduates.sum()*100,2)
    # Q8
    m=raw[(raw.high_school=='MISSION SAN JOSE HIGH SCHOOL')&(raw.fall_term==2023)&(raw.campus=='Universitywide')]
    a['Q8']=round(m.applicants.iloc[0]/m.ag_completers.iloc[0]*100,2)
    # Q9
    a['Q9']=int(raw[(raw.fall_term==2025)&(raw.campus=='Universitywide')&(raw.applicants>0)&(raw.school_type=='High Schools (Public)')].high_school.nunique())
    # Q10
    five=['HERCULES HIGH SCHOOL','MISSION SENIOR HIGH SCHOOL','MONTEREY TRAIL HIGH SCHOOL','PHILLIP & SALA BURTON ACAD HS','RANCHO SAN JUAN HIGH SCHOOL']
    bk=dash[(dash.campus=='Berkeley')&(dash.fall_term.between(2022,2025))&(dash.high_school.isin(five))]
    a['Q10']=bk.groupby('high_school').admit_rate_residual.mean().idxmax()
    return a

def method_sql(raw,dash,eth,trmaj,disc):
    con=sqlite3.connect(':memory:')
    raw.to_sql('schools',con,index=False); dash.to_sql('dash',con,index=False)
    eth.to_sql('eth',con,index=False); trmaj.to_sql('trmaj',con,index=False); disc.to_sql('disc',con,index=False)
    q=lambda x:con.execute(x).fetchall()
    a={}
    a['Q1']=round(q("SELECT CAST(SUM(CASE WHEN campus<>'Universitywide' THEN applicants END) AS REAL)/SUM(CASE WHEN campus='Universitywide' THEN applicants END) FROM schools WHERE fall_term=2025")[0][0],2)
    a['Q2']=round(q("SELECT CAST(SUM(admits) AS REAL)*100/SUM(applicants) FROM dash WHERE fall_term=2025 AND campus='Los Angeles' AND school_type='High Schools (Public)'")[0][0],2)
    a['Q3']=q("SELECT c.campus FROM disc c JOIN disc o ON c.campus=o.campus AND o.broad_discipline='All disciplines' WHERE c.fall_term=2025 AND c.broad_discipline='Computer Science' ORDER BY (c.admit_rate-o.admit_rate) ASC LIMIT 1")[0][0]
    a['Q4']=round(q("SELECT admit_gpa_p75-admit_gpa_p25 FROM trmaj WHERE campus='Berkeley' AND broad_discipline='Computer Science' AND major='ComputerScience'")[0][0],2)
    a['Q5']=sum(1 for r in q("SELECT campus, CAST(SUM(CASE WHEN ethnicity='White' AND count_type='Adm' THEN n END) AS REAL)/SUM(CASE WHEN ethnicity='White' AND count_type='App' THEN n END) wr, CAST(SUM(CASE WHEN ethnicity='Hispanic/Latino(a)' AND count_type='Adm' THEN n END) AS REAL)/SUM(CASE WHEN ethnicity='Hispanic/Latino(a)' AND count_type='App' THEN n END) hr FROM eth WHERE fall_term=2025 AND campus<>'Systemwide' GROUP BY campus") if r[1]>r[2])
    wa=q("SELECT SUM(n) FROM eth WHERE fall_term=2025 AND campus='Systemwide' AND ethnicity='White' AND count_type='Adm'")[0][0]; wp=q("SELECT SUM(n) FROM eth WHERE fall_term=2025 AND campus='Systemwide' AND ethnicity='White' AND count_type='App'")[0][0]
    ha=q("SELECT SUM(n) FROM eth WHERE fall_term=2025 AND campus='Systemwide' AND ethnicity='Hispanic/Latino(a)' AND count_type='Adm'")[0][0]; hp=q("SELECT SUM(n) FROM eth WHERE fall_term=2025 AND campus='Systemwide' AND ethnicity='Hispanic/Latino(a)' AND count_type='App'")[0][0]
    a['Q6']='Hispanic/Latino(a)' if ha/hp>wa/wp else 'White'
    bay="'"+"','".join(['Alameda','Contra Costa','Marin','Napa','San Francisco','San Mateo','Santa Clara','Solano','Sonoma'])+"'"; a['Q7']=round(q(f"SELECT CAST(SUM(enrolled_ccc) AS REAL)*100/SUM(graduates) FROM schools WHERE fall_term=2023 AND campus='Universitywide' AND county IN ({bay})")[0][0],2)
    a['Q8']=round(q("SELECT CAST(SUM(applicants) AS REAL)*100/SUM(ag_completers) FROM schools WHERE high_school='MISSION SAN JOSE HIGH SCHOOL' AND fall_term=2023 AND campus='Universitywide'")[0][0],2)
    a['Q9']=q("SELECT COUNT(DISTINCT high_school) FROM schools WHERE fall_term=2025 AND campus='Universitywide' AND applicants>0 AND school_type='High Schools (Public)'")[0][0]
    five="'"+"','".join(['HERCULES HIGH SCHOOL','MISSION SENIOR HIGH SCHOOL','MONTEREY TRAIL HIGH SCHOOL','PHILLIP & SALA BURTON ACAD HS','RANCHO SAN JUAN HIGH SCHOOL'])+"'"; a['Q10']=q(f"SELECT high_school FROM dash WHERE campus='Berkeley' AND fall_term BETWEEN 2022 AND 2025 AND high_school IN ({five}) GROUP BY high_school ORDER BY AVG(admit_rate_residual) DESC LIMIT 1")[0][0]
    return a

def method_manual(raw,dash,eth,trmaj,disc):
    a={}
    uw=camp=0
    for _,r in raw[raw.fall_term==2025].iterrows():
        if r.campus=='Universitywide': uw+=r.applicants
        else: camp+=r.applicants
    a['Q1']=round(camp/uw,2)
    ad=ap=0
    for _,r in dash.iterrows():
        if r.fall_term==2025 and r.campus=='Los Angeles' and r.school_type=='High Schools (Public)': ad+=r.admits; ap+=r.applicants
    a['Q2']=round(ad/ap*100,2)
    ov={};cs={}
    for _,r in disc.iterrows():
        if r.fall_term==2025:
            if r.broad_discipline=='All disciplines': ov[r.campus]=r.admit_rate
            if r.broad_discipline=='Computer Science': cs[r.campus]=r.admit_rate
    pen={c:(cs[c]-ov[c])*100 for c in cs}; a['Q3']=min(pen,key=pen.get)
    for _,r in trmaj.iterrows():
        if r.campus=='Berkeley' and r.broad_discipline=='Computer Science' and r.major=='ComputerScience': a['Q4']=round(r.admit_gpa_p75-r.admit_gpa_p25,2)
    cwa={};cwp={};cha={};chp={}
    for _,r in eth.iterrows():
        if r.fall_term==2025 and r.campus!='Systemwide':
            if r.ethnicity=='White': cwa[r.campus]=cwa.get(r.campus,0)+(r.n if r.count_type=='Adm' else 0); cwp[r.campus]=cwp.get(r.campus,0)+(r.n if r.count_type=='App' else 0)
            if r.ethnicity=='Hispanic/Latino(a)': cha[r.campus]=cha.get(r.campus,0)+(r.n if r.count_type=='Adm' else 0); chp[r.campus]=chp.get(r.campus,0)+(r.n if r.count_type=='App' else 0)
    a['Q5']=sum(1 for c in cwa if cwa[c]/cwp[c]>cha[c]/chp[c])
    wa=ha=wp=hp=0
    for _,r in eth.iterrows():
        if r.fall_term==2025 and r.campus=='Systemwide':
            if r.ethnicity=='White': wa+=r.n if r.count_type=='Adm' else 0; wp+=r.n if r.count_type=='App' else 0
            if r.ethnicity=='Hispanic/Latino(a)': ha+=r.n if r.count_type=='Adm' else 0; hp+=r.n if r.count_type=='App' else 0
    a['Q6']='Hispanic/Latino(a)' if ha/hp>wa/wp else 'White'
    bay={'Alameda','Contra Costa','Marin','Napa','San Francisco','San Mateo','Santa Clara','Solano','Sonoma'}; ccc=grad=0
    for _,r in raw.iterrows():
        if r.fall_term==2023 and r.campus=='Universitywide' and r.county in bay: ccc+=r.enrolled_ccc; grad+=r.graduates
    a['Q7']=round(ccc/grad*100,2)
    ap_=ag_=0
    for _,r in raw.iterrows():
        if r.high_school=='MISSION SAN JOSE HIGH SCHOOL' and r.fall_term==2023 and r.campus=='Universitywide': ap_+=r.applicants; ag_+=r.ag_completers
    a['Q8']=round(ap_/ag_*100,2)
    hs=set()
    for _,r in raw.iterrows():
        if r.fall_term==2025 and r.campus=='Universitywide' and r.applicants>0 and r.school_type=='High Schools (Public)': hs.add(r.high_school)
    a['Q9']=len(hs)
    five={'HERCULES HIGH SCHOOL','MISSION SENIOR HIGH SCHOOL','MONTEREY TRAIL HIGH SCHOOL','PHILLIP & SALA BURTON ACAD HS','RANCHO SAN JUAN HIGH SCHOOL'}; acc={}
    for _,r in dash.iterrows():
        if r.campus=='Berkeley' and 2022<=r.fall_term<=2025 and r.high_school in five: acc[r.high_school]=acc.get(r.high_school,[])+[r.admit_rate_residual]
    a['Q10']=max(acc,key=lambda s:sum(acc[s])/len(acc[s]))
    return a

In [ ]:
# Run 100 trials across the 3 methods (methods are deterministic; loop proves stability + catches any random-state bugs)
methods=[method_pandas, method_sql, method_manual]
trials=100
from collections import defaultdict
agg=defaultdict(lambda: defaultdict(list))
for t in range(trials):
    for m in methods:
        res=m(raw,dash,eth,trmaj,disc)
        for k,v in res.items(): agg[k][str(m.__name__)].append(v)
print(f'Ran {trials} trials x {len(methods)} methods = {trials*len(methods)} computations')
FINAL={}
for k in sorted(agg):
    consistent=True; vals=set()
    for meth, lst in agg[k].items():
        u=set(lst)
        if len(u)>1: consistent=False
        vals.update(u)
    FINAL[k]=(list(vals)[0] if consistent and len(vals)==1 else 'INCONSISTENT')
    print(f'{k}: {dict(agg[k])} -> {"OK "+str(FINAL[k]) if consistent else "CHECK"}')

In [ ]:
print('=== FINAL VERIFIED ANSWERS (100 trials, 3 methods each) ===')
for k in sorted(FINAL): print(f'{k}: {FINAL[k]}')